#  Telco Customer Churn — MLflow Experiment Tracking
**Notebook:** 04_mlflow_tracking.ipynb  
**Goal:** Log every experiment (default + tuned models), artifacts, and register the best model in MLflow Model Registry.

```
mlruns/
├── 0/  (Default)  
└── <experiment_id>/  Telco_Churn_Experiments
    ├── <run_id_rf>/
    ├── <run_id_lgbm>/
    ├── <run_id_xgb_default>/
    └── <run_id_xgb_tuned>/    ← registered as Production
```

## 1.  Install & Import

In [1]:
# Install MLflow if not already present
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'mlflow', '-q'])
print('MLflow installed ✅')

MLflow installed ✅


In [1]:
import os
import json
import time
import warnings
import joblib

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe inside notebooks
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.xgboost
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from mlflow import MlflowClient

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120})

print(f'MLflow version : {mlflow.__version__}')
print(f'XGBoost        : {xgb.__version__}')

MLflow version : 3.14.0
XGBoost        : 3.2.0


## 2.  Load Data & Pre-Trained Artefacts

In [2]:
PROCESSED  = '../data/processed/'
MODELS_DIR = '../models/'
MLFLOW_DIR = '../mlruns'          # local tracking URI
ARTIFACTS  = '../artifacts/'      # temp dir for figures
os.makedirs(ARTIFACTS, exist_ok=True)
os.makedirs(MLFLOW_DIR, exist_ok=True)

X_train = pd.read_csv(PROCESSED + 'X_train.csv')
X_test  = pd.read_csv(PROCESSED + 'X_test.csv')
y_train = pd.read_csv(PROCESSED + 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED + 'y_test.csv').squeeze()

neg, pos   = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos  = neg / pos
RANDOM_STATE = 42
N_FOLDS      = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# ── Load tuned params & threshold from notebook 03 ──────────────────────────
with open(MODELS_DIR + 'best_params.json') as f:
    tuned_params = json.load(f)

with open(MODELS_DIR + 'best_threshold.txt') as f:
    best_threshold = float(f.read().strip())

print(f'X_train : {X_train.shape}  |  X_test : {X_test.shape}')
print(f'scale_pos_weight : {scale_pos:.2f}')
print(f'Best threshold   : {best_threshold}')
print(f'Tuned params loaded : {len(tuned_params)} params')

X_train : (5634, 45)  |  X_test : (1409, 45)
scale_pos_weight : 2.77
Best threshold   : 0.6
Tuned params loaded : 13 params


## 3.  MLflow Configuration

In [3]:
import os
import pathlib
import mlflow

# 1. Bypass the MLflow file store maintenance mode exception
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Define your MLFLOW_DIR path (make sure this variable is defined above)
# MLFLOW_DIR = "mlruns" 

# 2. Convert the Windows path to a properly formatted local URI (file:///C:/...)
local_path = pathlib.Path(os.path.abspath(MLFLOW_DIR))
tracking_uri = local_path.as_uri()

# ── Point MLflow at local tracking server ────────────────────────────────────
mlflow.set_tracking_uri(tracking_uri)

EXPERIMENT_NAME = 'Telco_Churn_Experiments'
mlflow.set_experiment(EXPERIMENT_NAME)
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print(f'Tracking URI  : {mlflow.get_tracking_uri()}')
print(f'Experiment    : {EXPERIMENT_NAME}')
print(f'Experiment ID : {exp.experiment_id}')

Tracking URI  : file:///C:/Users/hp/Desktop/github/customer-churn-predictor/mlruns
Experiment    : Telco_Churn_Experiments
Experiment ID : 138005034149242259


## 4.  Helper Functions
Centralise all metric computation, artifact generation, and MLflow logging to keep each run DRY.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.1  Metrics
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(model, X_te, y_te, threshold=0.5):
    """Return dict of all evaluation metrics."""
    prob   = model.predict_proba(X_te)[:, 1]
    pred   = (prob >= threshold).astype(int)
    cv_auc = cross_val_score(model, X_train, y_train,
                             cv=skf, scoring='roc_auc', n_jobs=-1).mean()
    return {
        'test_auc_roc'  : float(roc_auc_score(y_te, prob)),
        'cv_auc_roc'    : float(cv_auc),
        'test_accuracy' : float(accuracy_score(y_te, pred)),
        'test_precision': float(precision_score(y_te, pred, zero_division=0)),
        'test_recall'   : float(recall_score(y_te, pred, zero_division=0)),
        'test_f1'       : float(f1_score(y_te, pred, zero_division=0)),
    }, prob


# ─────────────────────────────────────────────────────────────────────────────
# 4.2  Confusion Matrix artifact
# ─────────────────────────────────────────────────────────────────────────────
def save_confusion_matrix(model, X_te, y_te, name, threshold=0.5):
    prob = model.predict_proba(X_te)[:, 1]
    pred = (prob >= threshold).astype(int)
    cm   = confusion_matrix(y_te, pred)

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn']).plot(
        ax=ax, cmap='Blues', colorbar=False, values_format='d'
    )
    ax.set_title(f'Confusion Matrix — {name}', fontweight='bold')
    path = ARTIFACTS + f'cm_{name.lower().replace(" ","_")}.png'
    plt.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    return path


# ─────────────────────────────────────────────────────────────────────────────
# 4.3  Feature Importance artifact
# ─────────────────────────────────────────────────────────────────────────────
def save_feature_importance(model, feature_names, name, top_n=20):
    imp = model.feature_importances_
    df  = (pd.DataFrame({'feature': feature_names, 'importance': imp})
             .sort_values('importance', ascending=False)
             .head(top_n))

    colors = plt.cm.YlOrRd(np.linspace(0.35, 0.85, top_n)[::-1])
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(df['feature'][::-1], df['importance'][::-1],
            color=colors, edgecolor='white')
    ax.set_title(f'Top-{top_n} Feature Importances — {name}', fontweight='bold')
    ax.set_xlabel('Importance')
    path = ARTIFACTS + f'fi_{name.lower().replace(" ","_")}.png'
    plt.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    return path


# ─────────────────────────────────────────────────────────────────────────────
# 4.4  ROC Curve artifact
# ─────────────────────────────────────────────────────────────────────────────
def save_roc_curve(model, X_te, y_te, name, prob=None):
    if prob is None:
        prob = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, prob)
    auc = roc_auc_score(y_te, prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, lw=2.5, color='#DD8452',
            label=f'AUC = {auc:.4f}')
    ax.plot([0,1],[0,1],'k--', lw=1.2)
    ax.fill_between(fpr, tpr, alpha=0.08, color='#DD8452')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {name}', fontweight='bold')
    ax.legend(loc='lower right')
    path = ARTIFACTS + f'roc_{name.lower().replace(" ","_")}.png'
    plt.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    return path


# ─────────────────────────────────────────────────────────────────────────────
# 4.5  SHAP Summary Plot artifact
# ─────────────────────────────────────────────────────────────────────────────
def save_shap_plot(model, X_sample, name):
    try:
        import shap
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(X_sample)
        # For classifiers returning list, take the churn class
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]

        fig, ax = plt.subplots(figsize=(9, 7))
        shap.summary_plot(shap_vals, X_sample, plot_type='bar',
                          max_display=20, show=False)
        plt.title(f'SHAP Feature Importance — {name}', fontweight='bold')
        path = ARTIFACTS + f'shap_{name.lower().replace(" ","_")}.png'
        plt.tight_layout()
        plt.savefig(path, bbox_inches='tight')
        plt.close()
        return path
    except ImportError:
        print('  ⚠️  shap not installed — skipping SHAP plot')
        return None
    except Exception as e:
        print(f'  ⚠️  SHAP error: {e}')
        return None


# ─────────────────────────────────────────────────────────────────────────────
# 4.6  Master logging function
# ─────────────────────────────────────────────────────────────────────────────
def log_run(
    run_name, model, params, tags,
    threshold=0.5,
    log_shap=False,
    shap_sample_size=500,
    mlflow_model_flavor=None   # 'xgboost' | 'sklearn'
):
    """
    Full MLflow run: fit (if not already fitted), compute metrics,
    generate and log all artifacts, log model with signature.
    Returns (run_id, metrics_dict).
    """
    print(f'\n[{run_name}] Starting MLflow run …')
    feature_names = X_train.columns.tolist()

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        # ── Tags ──────────────────────────────────────────────────────────────
        mlflow.set_tags(tags)

        # ── Train ─────────────────────────────────────────────────────────────
        t0 = time.time()
        model.fit(X_train, y_train)
        train_time = round(time.time() - t0, 2)

        # ── Parameters ────────────────────────────────────────────────────────
        mlflow.log_params(params)
        mlflow.log_param('threshold',   threshold)
        mlflow.log_param('n_features',  X_train.shape[1])
        mlflow.log_param('train_rows',  X_train.shape[0])
        mlflow.log_param('cv_folds',    N_FOLDS)
        mlflow.log_param('random_state', RANDOM_STATE)

        # ── Metrics ───────────────────────────────────────────────────────────
        metrics, prob = compute_metrics(model, X_test, y_test, threshold)
        metrics['train_time_sec'] = train_time
        mlflow.log_metrics(metrics)

        # ── Artifacts ─────────────────────────────────────────────────────────
        # Confusion matrix
        cm_path = save_confusion_matrix(model, X_test, y_test, run_name, threshold)
        mlflow.log_artifact(cm_path, artifact_path='plots')

        # Feature importance
        fi_path = save_feature_importance(model, feature_names, run_name)
        mlflow.log_artifact(fi_path, artifact_path='plots')

        # ROC curve
        roc_path = save_roc_curve(model, X_test, y_test, run_name, prob)
        mlflow.log_artifact(roc_path, artifact_path='plots')

        # SHAP
        if log_shap:
            X_sample = X_train.sample(min(shap_sample_size, len(X_train)),
                                      random_state=RANDOM_STATE)
            shap_path = save_shap_plot(model, X_sample, run_name)
            if shap_path:
                mlflow.log_artifact(shap_path, artifact_path='plots')

        # ── Model logging with signature ──────────────────────────────────────
        signature   = infer_signature(X_train, model.predict_proba(X_train))
        input_example = X_train.head(3)

        if mlflow_model_flavor == 'xgboost':
            mlflow.xgboost.log_model(
                model,
                artifact_path='model',
                signature=signature,
                input_example=input_example
            )
        else:
            mlflow.lightgbm.log_model(
                model,
                artifact_path='model',
                signature=signature,
                input_example=input_example
            )

        print(f'  run_id        : {run_id}')
        print(f'  AUC-ROC (test): {metrics["test_auc_roc"]:.4f}')
        print(f'  AUC-ROC (CV)  : {metrics["cv_auc_roc"]:.4f}')
        print(f'  F1-Score      : {metrics["test_f1"]:.4f}')
        print(f'  Train time    : {train_time}s')

    return run_id, metrics


print('Helper functions defined ✅')

Helper functions defined ✅


## 5.  Run 1 — Random Forest (Default)


In [5]:
rf_params = {
    'n_estimators' : 300,
    'max_depth'    : None,
    'min_samples_split': 2,
    'min_samples_leaf' : 1,
    'max_features' : 'sqrt',
    'bootstrap'    : True,
    'class_weight' : 'balanced',   # handles imbalance
}

rf_tags = {
    'model_family'  : 'RandomForest',
    'tuning_stage'  : 'default',
    'dataset'       : 'Telco_Churn',
    'engineer'      : 'notebook_04',
}

rf_model = RandomForestClassifier(**rf_params, n_jobs=-1, random_state=RANDOM_STATE)

rf_run_id, rf_metrics = log_run(
    run_name           = 'RandomForest_Default',
    model              = rf_model,
    params             = rf_params,
    tags               = rf_tags,
    mlflow_model_flavor= 'sklearn'
)


[RandomForest_Default] Starting MLflow run …


2026/06/25 12:18:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  run_id        : ea7aa610e67346eeb8a48741ecdbc32a
  AUC-ROC (test): 0.8254
  AUC-ROC (CV)  : 0.8283
  F1-Score      : 0.5441
  Train time    : 1.24s


## 6.  Run 2 — LightGBM (Default)

In [9]:
import os

# Allow MLflow/skops to trust all types globally during serialization
os.environ["MLFLOW_SKOPS_ALLOW_UNTRUSTED_TYPES"] = "True"

# Your original code follows:
lgbm_params = {
    'n_estimators'    : 300,
    'max_depth'       : -1,
    'learning_rate'   : 0.1,
    'num_leaves'      : 31,
    'subsample'       : 1.0,
    'colsample_bytree': 1.0,
    'reg_alpha'       : 0.0,
    'reg_lambda'      : 0.0,
    'scale_pos_weight': round(float(scale_pos), 4),
}


lgbm_tags = {
    'model_family': 'LightGBM',
    'tuning_stage': 'default',
    'dataset'     : 'Telco_Churn',
    'engineer'    : 'notebook_04',
}

lgbm_model = lgb.LGBMClassifier(**lgbm_params, verbosity=-1, random_state=RANDOM_STATE)


lgbm_run_id, lgbm_metrics = log_run(
    run_name           = 'LightGBM_Default',
    model              = lgbm_model,
    params             = lgbm_params,
    tags               = lgbm_tags,
    mlflow_model_flavor= 'lightgbm'  # Changed from 'sklearn'
)


[LightGBM_Default] Starting MLflow run …


2026/06/25 12:25:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  run_id        : 4715379c82584628bcab96cc121fea5a
  AUC-ROC (test): 0.8230
  AUC-ROC (CV)  : 0.8280
  F1-Score      : 0.5946
  Train time    : 0.36s


## 7.  Run 3 — XGBoost (Default)

In [10]:
xgb_default_params = {
    'n_estimators'    : 300,
    'max_depth'       : 6,
    'learning_rate'   : 0.1,
    'subsample'       : 1.0,
    'colsample_bytree': 1.0,
    'gamma'           : 0.0,
    'reg_alpha'       : 0.0,
    'reg_lambda'      : 1.0,
    'min_child_weight': 1,
    'scale_pos_weight': round(float(scale_pos), 4),
}

xgb_default_tags = {
    'model_family': 'XGBoost',
    'tuning_stage': 'default',
    'dataset'     : 'Telco_Churn',
    'engineer'    : 'notebook_04',
}

xgb_default_model = xgb.XGBClassifier(
    **xgb_default_params,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0,
    random_state=RANDOM_STATE
)

xgb_default_run_id, xgb_default_metrics = log_run(
    run_name           = 'XGBoost_Default',
    model              = xgb_default_model,
    params             = xgb_default_params,
    tags               = xgb_default_tags,
    mlflow_model_flavor= 'xgboost'
)


[XGBoost_Default] Starting MLflow run …


2026/06/25 12:25:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  run_id        : 0e3da9c606c94f48a4f64bf09317e4ab
  AUC-ROC (test): 0.8220
  AUC-ROC (CV)  : 0.8268
  F1-Score      : 0.5892
  Train time    : 0.6s


## 8.  Run 4 — XGBoost (Tuned) ← Best Model

In [12]:
# ── Rebuild tuned params (from notebook 03 JSON) ─────────────────────────────
xgb_tuned_params = {
    k: v for k, v in tuned_params.items()
    if k not in ('use_label_encoder', 'eval_metric', 'verbosity', 'n_jobs')
}
# Ensure scale_pos_weight is up-to-date
xgb_tuned_params['scale_pos_weight'] = round(float(scale_pos), 4)

xgb_tuned_tags = {
    'model_family'  : 'XGBoost',
    'tuning_stage'  : 'tuned_optuna_gridsearch',
    'dataset'       : 'Telco_Churn',
    'engineer'      : 'notebook_04',
    'best_model'    : 'true',
    'cv_strategy'   : f'StratifiedKFold_{N_FOLDS}fold',
    'tuning_method' : 'Optuna_TPE_100trials+GridSearchCV',
}

xgb_tuned_model = xgb.XGBClassifier(
    **xgb_tuned_params,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1
)
xgb_tuned_run_id, xgb_tuned_metrics = log_run(
    run_name           = 'XGBoost_Tuned',
    model              = xgb_tuned_model,
    params             = xgb_tuned_params,
    tags               = xgb_tuned_tags,
    threshold          = best_threshold,
    log_shap           = True,          # SHAP plot for best model only
    shap_sample_size   = 500,
    mlflow_model_flavor= 'xgboost'
)

print(f'\nBest model run_id: {xgb_tuned_run_id}')


[XGBoost_Tuned] Starting MLflow run …
  ⚠️  shap not installed — skipping SHAP plot


2026/06/25 12:28:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  run_id        : 9393605e89034f4c91e707d817856c97
  AUC-ROC (test): 0.8487
  AUC-ROC (CV)  : 0.8503
  F1-Score      : 0.6372
  Train time    : 0.66s

Best model run_id: 9393605e89034f4c91e707d817856c97


## 9.  Cross-Run Comparison

In [13]:
# ── Build comparison DataFrame from logged metrics ────────────────────────────
runs_summary = pd.DataFrame([
    {'Run': 'RandomForest Default',  **rf_metrics},
    {'Run': 'LightGBM Default',      **lgbm_metrics},
    {'Run': 'XGBoost Default',       **xgb_default_metrics},
    {'Run': 'XGBoost Tuned ',      **xgb_tuned_metrics},
]).set_index('Run').drop(columns=['train_time_sec'], errors='ignore')

styled = (
    runs_summary.style
    .format('{:.4f}')
    .highlight_max(color='#c6efce')
    .highlight_min(color='#ffc7ce')
    .set_caption('All Runs — Test Set Metrics (green = best, red = worst)')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size','14px'),('font-weight','bold')]}])
)
print(runs_summary.to_string())
styled

                      test_auc_roc  cv_auc_roc  test_accuracy  test_precision  test_recall   test_f1
Run                                                                                                 
RandomForest Default      0.825429    0.828341       0.783534        0.616949     0.486631  0.544096
LightGBM Default          0.822982    0.828005       0.765791        0.550000     0.647059  0.594595
XGBoost Default           0.822007    0.826813       0.756565        0.533623     0.657754  0.589222
XGBoost Tuned             0.848715    0.850335       0.784244        0.575431     0.713904  0.637232


,test_auc_roc,cv_auc_roc,test_accuracy,test_precision,test_recall,test_f1
Run,,,,,,
RandomForest Default,0.8254,0.8283,0.7835,0.6169,0.4866,0.5441
LightGBM Default,0.8230,0.8280,0.7658,0.5500,0.6471,0.5946
XGBoost Default,0.8220,0.8268,0.7566,0.5336,0.6578,0.5892
XGBoost Tuned,0.8487,0.8503,0.7842,0.5754,0.7139,0.6372


In [14]:
# ── Visual comparison ─────────────────────────────────────────────────────────
metric_cols = ['test_auc_roc', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1']
labels      = ['AUC-ROC', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
palette     = ['#4C72B0', '#55A868', '#DD8452', '#C44E52']
run_labels  = ['RF Default', 'LGBM Default', 'XGB Default', 'XGB Tuned ⭐']

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for ax, col, label in zip(axes, metric_cols, labels):
    vals = runs_summary[col].values
    bars = ax.bar(run_labels, vals, color=palette, edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    # Highlight best
    best_idx = int(np.argmax(vals))
    bars[best_idx].set_edgecolor('#2d6a4f')
    bars[best_idx].set_linewidth(2.5)
    ax.set_title(label, fontweight='bold')
    ax.set_ylim(min(vals)*0.94, 1.02)
    ax.tick_params(axis='x', rotation=30, labelsize=8)

plt.suptitle('All MLflow Runs — Test Set Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

comparison_fig_path = ARTIFACTS + 'all_runs_comparison.png'
plt.savefig(comparison_fig_path, bbox_inches='tight')
plt.show()

# Log comparison figure to the tuned model run
with mlflow.start_run(run_id=xgb_tuned_run_id):
    mlflow.log_artifact(comparison_fig_path, artifact_path='plots')
print('Comparison chart logged to XGBoost_Tuned run ✅')

Comparison chart logged to XGBoost_Tuned run ✅


## 10.  Model Registry — Register Best Model
Register `XGBoost_Tuned` in the MLflow Model Registry and promote it to **Production**.

In [15]:
REGISTRY_NAME = 'TelcoChurn_XGBoost_Best'
client        = MlflowClient()

# ── 10.1  Register model ──────────────────────────────────────────────────────
model_uri = f'runs:/{xgb_tuned_run_id}/model'

registered = mlflow.register_model(
    model_uri   = model_uri,
    name        = REGISTRY_NAME
)
version = registered.version
print(f'Registered model : {REGISTRY_NAME}  version {version}')

Successfully registered model 'TelcoChurn_XGBoost_Best'.
2026/06/25 12:29:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 9393605e89034f4c91e707d817856c97 has no artifacts at artifact path 'model', registering model based on models:/m-72923189e89542139e6e0a637ac51205 instead


Registered model : TelcoChurn_XGBoost_Best  version 1


Created version '1' of model 'TelcoChurn_XGBoost_Best'.


In [16]:
# ── 10.2  Wait for registration to complete ───────────────────────────────────
import time as _time
for _ in range(20):
    mv = client.get_model_version(REGISTRY_NAME, version)
    if mv.status == 'READY':
        print(f'Model version {version} status: READY ✅')
        break
    _time.sleep(1)
else:
    print(f'Model version {version} status: {mv.status}')

Model version 1 status: READY ✅


In [17]:
# ── 10.3  Add description & tags to registered model ─────────────────────────
client.update_registered_model(
    name        = REGISTRY_NAME,
    description = (
        'XGBoost classifier tuned with Optuna TPE (100 trials) + GridSearchCV '
        'on Telco Customer Churn dataset. '
        f'Test AUC-ROC: {xgb_tuned_metrics["test_auc_roc"]:.4f}. '
        f'CV AUC-ROC ({N_FOLDS}-fold): {xgb_tuned_metrics["cv_auc_roc"]:.4f}. '
        f'Decision threshold: {best_threshold}.'
    )
)

client.update_model_version(
    name        = REGISTRY_NAME,
    version     = version,
    description = f'Best model — Optuna + GridSearch. AUC={xgb_tuned_metrics["test_auc_roc"]:.4f}'
)

# Set version-level tags
for key, val in {
    'auc_roc'         : f'{xgb_tuned_metrics["test_auc_roc"]:.4f}',
    'f1_score'        : f'{xgb_tuned_metrics["test_f1"]:.4f}',
    'threshold'       : str(best_threshold),
    'tuning'          : 'Optuna_100trials+GridSearchCV',
    'cv_strategy'     : f'StratifiedKFold_{N_FOLDS}fold',
}.items():
    client.set_model_version_tag(REGISTRY_NAME, version, key, val)

print('Description and tags added ✅')

Description and tags added ✅


In [18]:
# ── 10.4  Transition to Production ───────────────────────────────────────────
client.transition_model_version_stage(
    name    = REGISTRY_NAME,
    version = version,
    stage   = 'Production',
    archive_existing_versions=True   # archive any previous Production version
)

mv = client.get_model_version(REGISTRY_NAME, version)
print(f'\n╔═══════════════════════════════════════════════════╗')
print(f'║  Model   : {REGISTRY_NAME:<37}║')
print(f'║  Version : {version:<37}║')
print(f'║  Stage   : {mv.current_stage:<37}║')
print(f'║  AUC-ROC : {xgb_tuned_metrics["test_auc_roc"]:.4f}{"":32}║')
print(f'╚═══════════════════════════════════════════════════╝')


╔═══════════════════════════════════════════════════╗
║  Model   : TelcoChurn_XGBoost_Best              ║
║  Version : 1                                    ║
║  Stage   : Production                           ║
║  AUC-ROC : 0.8487                                ║
╚═══════════════════════════════════════════════════╝


## 11.  Verify: Load Model from Registry & Score

In [21]:
# ── Load straight from registry ───────────────────────────────────────────────
prod_uri   = f'models:/{REGISTRY_NAME}/Production'
prod_model = mlflow.xgboost.load_model(prod_uri)

prob_check = prod_model.predict_proba(X_test)[:, 1]
auc_check  = roc_auc_score(y_test, prob_check)

print(f'Production model loaded from: {prod_uri}')
print(f'AUC-ROC (re-scored)          : {auc_check:.4f}')
print(f'Match original metrics       : {abs(auc_check - xgb_tuned_metrics["test_auc_roc"]) < 1e-6}')

Production model loaded from: models:/TelcoChurn_XGBoost_Best/Production
AUC-ROC (re-scored)          : 0.8487
Match original metrics       : True


## 12.  Launch MLflow UI

In [22]:
# ── Print all run IDs for quick reference ─────────────────────────────────────
run_map = {
    'RandomForest_Default' : rf_run_id,
    'LightGBM_Default'     : lgbm_run_id,
    'XGBoost_Default'      : xgb_default_run_id,
    'XGBoost_Tuned'        : xgb_tuned_run_id,
}
print('Run IDs:')
for name, rid in run_map.items():
    print(f'  {name:<25}: {rid}')

print(f'\nExperiment  : {EXPERIMENT_NAME}')
print(f'Registry    : {REGISTRY_NAME}  (v{version} → Production)')
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  To launch the MLflow UI, run in your terminal:

    cd <project_root>
    mlflow ui --backend-store-uri mlruns --port 5000

  Then open: http://127.0.0.1:5000

  What to check in the UI:
    • Experiment tab   → 4 runs, sortable by AUC-ROC
    • Each run         → Params, Metrics, Artifacts (plots/)
    • Models tab       → TelcoChurn_XGBoost_Best  v{version}  Production
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""".format(version=version))

Run IDs:
  RandomForest_Default     : ea7aa610e67346eeb8a48741ecdbc32a
  LightGBM_Default         : 4715379c82584628bcab96cc121fea5a
  XGBoost_Default          : 0e3da9c606c94f48a4f64bf09317e4ab
  XGBoost_Tuned            : 9393605e89034f4c91e707d817856c97

Experiment  : Telco_Churn_Experiments
Registry    : TelcoChurn_XGBoost_Best  (v1 → Production)
Tracking URI: file:///C:/Users/hp/Desktop/github/customer-churn-predictor/mlruns

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  To launch the MLflow UI, run in your terminal:

    cd <project_root>
    mlflow ui --backend-store-uri mlruns --port 5000

  Then open: http://127.0.0.1:5000

  What to check in the UI:
    • Experiment tab   → 4 runs, sortable by AUC-ROC
    • Each run         → Params, Metrics, Artifacts (plots/)
    • Models tab       → TelcoChurn_XGBoost_Best  v1  Production
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [ ]:
# ── Optional: launch UI directly from notebook (runs in background) ───────────
# Uncomment to auto-launch (blocks if you don't send to background)

# import subprocess
# proc = subprocess.Popen(
#     ['mlflow', 'ui',
#      '--backend-store-uri', os.path.abspath(MLFLOW_DIR),
#      '--port', '5000'],
#     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
# )
# print(f'MLflow UI started (PID {proc.pid}) → http://127.0.0.1:5000')
# print('Call proc.terminate() to stop it.')

## 13.  Summary

| Step | Detail |
|------|--------|
| Tracking URI | `file://<project_root>/mlruns` (local) |
| Experiment | `Telco_Churn_Experiments` |
| Runs logged | RandomForest Default · LightGBM Default · XGBoost Default · XGBoost Tuned |
| Per-run artifacts | Confusion matrix · Feature importance · ROC curve · SHAP (tuned only) |
| Per-run logs | Params · Metrics (test + 5-fold CV) · Model with signature + input example |
| Model Registry | `TelcoChurn_XGBoost_Best` v1 → **Production** |
| UI launch | `mlflow ui --backend-store-uri mlruns --port 5000` |

### What each artifact tells you

| Artifact | Location | Purpose |
|----------|----------|---------|
| `cm_*.png` | `plots/` | Visualise TP/TN/FP/FN at chosen threshold |
| `fi_*.png` | `plots/` | Which features the model relies on |
| `roc_*.png` | `plots/` | Full AUC curve — threshold-independent performance |
| `shap_*.png` | `plots/` (tuned only) | Feature impact direction + magnitude |
| `all_runs_comparison.png` | `plots/` (tuned run) | Cross-run metric bar chart |

> ▶️ **Next step:** `05_inference.ipynb` — load the Production model from the registry and score new customers.